# IMDB Project - Chatbot_IMDB

In [1]:
# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/master/notebooks/colab-github-demo.ipynb)


## IMDB genre translation dictionary

In [2]:
GENRES_IMDB = {
    28: "Action",
    12: "Adventure",
    16: "Animation",
    35: "Comedy",
    80: "Crime",
    99: "Documentary",
    18: "Drama",
    10751: "Family",
    14: "Fantasy",
    36: "History",
    27: "Horror",
    10402: "Music",
    9648: "Mystery",
    10749: "Romance",
    878: "Sci-Fi",
    10770: "TV Movie",
    53: "Thriller",
    10752: "War",
    37: "Western"
}

GENRES_IMDB_INVERTED = {
    "Action": 28,
    "Adventure": 12,
    "Animation": 16,
    "Comedy": 35,
    "Crime": 80,
    "Documentary": 99,
    "Drama": 18,
    "Family": 10751,
    "Fantasy": 14,
    "History": 36,
    "Horror": 27,
    "Music": 10402,
    "Mystery": 9648,
    "Romance": 10749,
    "Sci-Fi": 878,
    "TV Movie": 10770,
    "Thriller": 53,
    "War": 10752,
    "Western": 37
}



# Data Scraping with API

In [ ]:
from pathlib import Path

import requests
import json
import time
import pandas as pd
import os

# API configuration.
API_KEY = "c436a0598ba40f517d94fa3c9cc217d6"  # Replace with your TMDB API key.
BASE_URL = "https://api.themoviedb.org/3/movie/popular"
NUM_MOVIES = 10000  # Total number of movies to download.
MOVIES_PER_PAGE = 20  # TMDB returns 20 movies per page.
pages_to_download = (NUM_MOVIES // MOVIES_PER_PAGE) + 1

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
output_path = data_dir / "popular_movies_10k.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

def get_movies(total=NUM_MOVIES):
    movies = []
    for page in range(1, pages_to_download + 1):
        url = f"{BASE_URL}?api_key={API_KEY}&language=en-EN&page={page}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            movies.extend(data["results"])
        else:
            print(f"⚠ Request error: {response.status_code}")
            break
        # Pause to avoid exceeding API limits (40 requests/10s).
        time.sleep(0.1)
        # Stop once the requested limit is reached.
        if len(movies) >= total:
            break

    return movies[:total]

# Fetch the most popular movies.
movies = get_movies(NUM_MOVIES)

# Create a DataFrame and save it as a CSV file.
movies_df = pd.DataFrame(movies)

movies_df.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Saved {len(movies)} movies to {output_path}")

✅ Se han guardado 10000 películas en ./peliculasPopulares10k.csv


# Dataset Cleaning

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
input_path = data_dir / "popular_movies_10k.csv"

# Read the CSV file in chunks.
chunks = pd.read_csv(input_path, encoding="utf-8", sep=",", chunksize=1000)

# Concatenate chunks into a single DataFrame.
movies_df = pd.concat(chunks, ignore_index=True)

columns = ["title", "release_date", "popularity", "original_language", "overview", "genre_ids", "adult"]
movies_df = movies_df[columns]

# Remove duplicate rows based on all columns.
movies_df.drop_duplicates(inplace=True)

# Remove rows with missing values (NaN) in any column.
movies_df.dropna(inplace=True)

# Reset the index after removing rows.
movies_df.reset_index(drop=True, inplace=True)

movies_df.head()

processed_path = data_dir / "popular_movies_10k_processed.csv"
# Save the cleaned DataFrame to a CSV file.
movies_df.to_csv(processed_path, index=False, encoding="utf-8")
print(f"✅ CSV file saved to {processed_path}")

✅ Archivo CSV guardado en ./peliculasPopulares10k_Procesado.csv


## Filter Functions

In [ ]:
def get_genre_dataframe(genre):
  if isinstance(genre, str):
    genre = GENRES_IMDB_INVERTED[genre]
  genre_df = movies_df[movies_df['genre_ids'].apply(lambda value: genre in value)]
  return genre_df

def get_language_dataframe(language):
  language_df = movies_df[movies_df['original_language'] == language]
  return language_df

def get_year_range_dataframe(start_year, end_year):
  """
  Filter the DataFrame to include movies released within a specified year range.

  Args:
    start_year: The starting year of the range (inclusive).
    end_year: The ending year of the range (inclusive).

  Returns:
    A filtered DataFrame containing movies released within the specified range.
  """
  year_range_df = movies_df[
      movies_df['release_date'].str.slice(0, 4).between(str(start_year), str(end_year))
  ]
  return year_range_df

# Example usage
movies_2020_to_2023 = get_year_range_dataframe(2020, 2023)
# You can now work with the movies_2020_to_2023 DataFrame

movies_2020_to_2023.head()

,title,release_date,popularity,original_language,overview,genre_ids,adult
42,My Fault,2023-06-08,442.550,es,"Noah must leave her city, boyfriend, and frien...","[10749, 18]",False
55,Sonic the Hedgehog 2,2022-03-30,305.582,en,"After settling in Green Hills, Sonic is eager ...","[28, 12, 10751, 35]",False
57,Sex Game 6969,2022-01-27,338.296,ko,Three married women had always been dissatisfi...,"[35, 18, 10749]",False
78,Fast X,2023-05-17,250.448,en,Over many missions and against impossible odds...,"[28, 80, 53]",False
83,365 Days: This Day,2022-04-26,231.718,pl,Laura and Massimo are back and hotter than eve...,"[10749, 18]",False


## Building the Dense Retriever Class

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path
import os

class DenseRetriever:
    def __init__(self, df, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        """
        Initialize the embedding model and keep embeddings in memory.
        :param df: DataFrame with the columns ["title", "overview"].
        :param model_name: Hugging Face model name.
        """
        self.df = df
        self.model = SentenceTransformer(model_name)
        self.embeddings = None

        # Concatenate title and overview to generate embeddings.
        self.df["text"] = self.df["title"] + " - " + self.df["overview"]

        # Generate embeddings for the movies.
        project_root = Path.cwd()
        if not (project_root / "data").exists():
            project_root = project_root.parent
        self._generate_embeddings(project_root / "data" / "movie_embeddings.npy")

    def _generate_embeddings(self, path_embeddings=None):
        """Generate embeddings and keep them in memory."""
        if path_embeddings and os.path.exists(path_embeddings):
            self.embeddings = np.load(path_embeddings)
            print(f"Embeddings loaded from: {path_embeddings}")
        else:
          print("🔹 Generating embeddings...")
          self.embeddings = self.model.encode(self.df["text"].tolist(), convert_to_numpy=True)
          np.save(path_embeddings if path_embeddings else "movie_embeddings.npy", self.embeddings)
          print("✅ Embeddings generated.")

    def save_embeddings(self, path):
        """Save embeddings to a .npy file."""
        np.save(path, self.embeddings)
        print(f"Embeddings saved to: {path}")


    def search(self, query, top_k=5):
        """
        Perform a cosine-similarity search.
        :param query: Search text.
        :param top_k: Number of results to return.
        :return: DataFrame with results sorted by similarity.
        """
        print(f"🔍 Searching for: {query}")

        # Convert the query into an embedding.
        query_embedding = self.model.encode([query], convert_to_numpy=True)

        # Compute cosine similarity between the query and movie embeddings.
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]

        # Retrieve the best result indices.
        best_indices = np.argsort(similarities)[::-1][:top_k]

        # Collect the matching movies.
        results = self.df.iloc[best_indices].copy()
        results["similarity"] = similarities[best_indices]

        return results.sort_values(by="similarity", ascending=False)

## Embedding Preparation

### Load the processed dataset

In [ ]:
movies_df = pd.read_csv(processed_path)

### Load the Dense Retriever class and generate embeddings for the dataframe

In [ ]:
dense_retriever = DenseRetriever(movies_df)

🔹 Generando embeddings...
✅ Embeddings generados.


## Dense search test based on a query

In [ ]:
query = "a kid who learns kung fu from an old master in China"
results = dense_retriever.search(query, 5)
results

🔍 Buscando: a kid who learns kung fu with a old man on China


,title,release_date,popularity,original_language,overview,genre_ids,adult,text,similarity
2094,Karate Kid: Legends,2025-05-28,40.495,en,"After a family tragedy, kung fu prodigy Li Fon...","[28, 18, 10751]",False,"Karate Kid: Legends - After a family tragedy, ...",0.691103
1327,Karate Kid: Legends,2025-05-28,40.495,en,"After a family tragedy, kung fu prodigy Li Fon...","[28, 18, 10751]",False,"Karate Kid: Legends - After a family tragedy, ...",0.691103
6475,Man of Tai Chi,2013-07-04,15.281,en,"In Beijing, a young martial artist's skill pla...","[28, 18]",False,"Man of Tai Chi - In Beijing, a young martial a...",0.597481
4962,Kung Fu Jungle,2014-10-31,18.453,zh,A martial arts instructor working at a police ...,"[28, 53, 80, 12]",False,Kung Fu Jungle - A martial arts instructor wor...,0.589305
90,Kung Fu Panda 4,2024-03-02,215.393,en,Po is gearing up to become the spiritual leade...,"[16, 10751, 14, 28]",False,Kung Fu Panda 4 - Po is gearing up to become t...,0.573977


In [ ]:
query = "A weapons manufacturer is kidnapped and becomes an armored superhero"
results = dense_retriever.search(query, 10)
results

🔍 Buscando: A weapons businessman is kidnapped and becomes an armored superhero


,title,release_date,popularity,original_language,overview,genre_ids,adult,text,similarity
3769,Super,2010-11-26,20.506,en,After his wife falls under the influence of a ...,"[35, 28, 18]",False,Super - After his wife falls under the influen...,0.587949
299,Iron Man,2008-04-30,100.273,en,"After being held captive in an Afghan cave, bi...","[28, 878, 12]",False,Iron Man - After being held captive in an Afgh...,0.563994
6262,The Great Arms Robbery,2022-04-09,14.700,zh,Agent Wen goes undercover to locate weapons fo...,"[28, 80, 18]",False,The Great Arms Robbery - Agent Wen goes underc...,0.546860
2097,Commando,1985-10-03,28.813,en,"John Matrix, the former leader of a special co...","[28, 12, 53]",False,"Commando - John Matrix, the former leader of a...",0.492411
72,Armor,2024-10-30,252.688,en,Armored truck security guard James Brody is wo...,"[28, 80, 53, 18]",False,Armor - Armored truck security guard James Bro...,0.480697
397,Iron Man 2,2010-04-28,83.981,en,With the world now aware of his dual life as t...,"[12, 28, 878]",False,Iron Man 2 - With the world now aware of his d...,0.472264
3007,Ransom,1996-11-08,24.251,en,"When a rich man's son is kidnapped, he coopera...","[28, 53]",False,"Ransom - When a rich man's son is kidnapped, h...",0.469037
5350,6 Bullets,2012-09-11,16.150,en,An ex-mercenary known for finding missing chil...,"[53, 28, 80]",False,6 Bullets - An ex-mercenary known for finding ...,0.468657
4640,Iron Man & Captain America: Heroes United,2014-07-29,17.686,en,Iron Man and Captain America battle to keep th...,"[12, 16, 28]",False,Iron Man & Captain America: Heroes United - Ir...,0.467726
2179,Watchmen: Chapter I,2024-08-12,29.722,en,"In 1985, the murder of a government-sponsored ...","[16, 18, 878]",False,"Watchmen: Chapter I - In 1985, the murder of a...",0.467210
